In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn scipy sqlalchemy joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

RANDOM_STATE = 42

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Libraries imported successfully!
Pandas version: 3.1.0
NumPy version: 2.5.4


#### Download the Dataset


In [9]:


DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

data_url = (
    "https://raw.githubusercontent.com/"
    "IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

file_path = DATA_DIR / "Telco-Customer-Churn.csv"

if not file_path.exists():
    df = pd.read_csv(data_url)
    df.to_csv(file_path, index=False)
else:
    df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#### Understand the dataset

In [4]:
print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

Rows and columns: (7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Dataset informat

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("Duplicate rows:", df.duplicated().sum())

if "customerID" in df.columns:
    print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

print("Target distribution:")
display(df["Churn"].value_counts(dropna=False))

print("Target proportions:")
display(
    df["Churn"].value_counts(normalize=True, dropna=False) * 100
)

Missing values:


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Duplicate rows: 0
Duplicate customer IDs: 0
Target distribution:


Churn
No     5174
Yes    1869
Name: count, dtype: int64

Target proportions:


Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

In [6]:
df = df.drop_duplicates().copy()

# Convert TotalCharges to a numeric column.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Inspect missing values before deciding how to handle them.
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

display(
    df.loc[
        df["TotalCharges"].isna(),
        ["customerID", "tenure", "TotalCharges"]
    ].head(15)
)

Missing TotalCharges: 11


,customerID,tenure,TotalCharges
488,4472-LVYGI,0,NaN
753,3115-CZMZD,0,NaN
936,5709-LVOEQ,0,NaN
1082,4367-NUYAO,0,NaN
1340,1371-DWPAZ,0,NaN
3331,7644-OMVMY,0,NaN
3826,3213-VVOLG,0,NaN
4380,2520-SGTTA,0,NaN
5218,2923-ARZLG,0,NaN
6670,4075-WKNIU,0,NaN


In [7]:
zero_tenure_missing = (
    df["TotalCharges"].isna() & (df["tenure"] == 0)
)

print("Missing values with zero tenure:", zero_tenure_missing.sum())

# Use zero only when the record indicates no accumulated tenure
# and no total charges have been recorded.
df.loc[zero_tenure_missing, "TotalCharges"] = 0

# Any remaining missing values must be investigated.
print("Remaining missing values:")
display(df.isnull().sum()[df.isnull().sum() > 0])

Missing values with zero tenure: 11
Remaining missing values:


Series([], dtype: int64)

In [8]:
df["ChurnFlag"] = (
    df["Churn"].astype(str).str.strip().str.lower().map(
        {"yes": 1, "no": 0}
    )
)

if df["ChurnFlag"].isna().any():
    raise ValueError("Unexpected or missing churn labels detected.")

churn_rate = df["ChurnFlag"].mean() * 100

print(f"Overall customer churn rate: {churn_rate:.2f}%")

print("\nChurn by contract type:")
display(
    df.groupby("Contract", dropna=False)
      .agg(
          customers=("ChurnFlag", "size"),
          churn_rate=("ChurnFlag", "mean"),
          average_monthly_charges=("MonthlyCharges", "mean")
      )
      .assign(churn_rate=lambda x: x["churn_rate"] * 100)
      .round(2)
      .sort_values("churn_rate", ascending=False)
)

Overall customer churn rate: 26.54%

Churn by contract type:


,customers,churn_rate,average_monthly_charges
Contract,,,
Month-to-month,3875,42.71,66.40
One year,1473,11.27,65.05
Two year,1695,2.83,60.77
